# Locally Adapted Conformal Prediction

In [1]:
MC_simulation_samples = 1000
n_models_ensemble = 1000
alpha = 0.05

dataset_name_list = ["train", "test", "calib"]

valor_max_list = [10]*3
valor_min_list = [0]*3
num_amostras_list = [100, 1000, 0]

erro_sistematico_x_list = [0.0]*3
erro_aleatorio_x_list = [0.2]*3
dist_erro_x_list = ["uniforme"]*3

erro_sistematico_y_list = [0]*3
erro_aleatorio_y_list = [0.1]*3
dist_erro_y_list = ["uniforme"]*3

gerar_mc_list = [True, False, False]

## Ambiente Básico

#### Imports

In [2]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import time
import optuna

from modelo_scr.utils import *

from modelo_scr.modelo_pacheco import *
from modelo_scr.nn_ensamble import *
from modelo_scr.rf_ensamble import *
from modelo_scr.conformal_prediction import *

from sklearn.metrics import mean_absolute_error

#### Modelo do sistema simuado

In [3]:
def modelo(x):
    return 10*x**2

#### Gera os dataset

In [4]:
dataset = {}

for i in range(len(dataset_name_list)):
    print(f"Gerando dataset {dataset_name_list[i]}:")
    dataset[dataset_name_list[i]] = Dataset(modelo, 
                                            valor_max_list[i], 
                                            valor_min_list[i], 
                                            num_amostras_list[i],
                                            erro_sistematico_x_list[i], 
                                            erro_aleatorio_x_list[i], 
                                            dist_erro_x_list[i], 
                                            erro_sistematico_y_list[i], 
                                            erro_aleatorio_y_list[i], 
                                            dist_erro_y_list[i])
    if gerar_mc_list[i]:
        dataset[dataset_name_list[i]].gerar_monte_carlo(MC_simulation_samples)


Gerando dataset train:
Shape X_measured: (100,)
Shape y_measured: (100,)
Shape X_measured_mc: (100000,)
Shape y_measured_mc: (100000,)
Gerando dataset test:
Shape X_measured: (1000,)
Shape y_measured: (1000,)
Gerando dataset calib:
Shape X_measured: (0,)
Shape y_measured: (0,)


## Treinamento da NN Ensable

In [5]:
# Definição do modelo
def model_fn():
    return MLP(input_dim=1, hidden_layers=[72]*2, activation="relu", dropout=0.0003)

# Configuração
config = TrainerConfig(epochs=500, lr=1e-3)

Usando dispositivo: cuda


### Modelo treinado com train dataset
Utiliza os hiperparâmetros definidos na otimização do modelo do pacheco uma vez que a arquitetura é a mesma, só que sem o ensable de incerteza.

In [6]:
modelo_nn = NNEnsambleModel(model_fn,
                config,
                n_models=n_models_ensemble,
                verbose=True)

modelo_nn.fit(dataset["train"].X_measured.reshape((-1, 1)).numpy(), 
              dataset["train"].y_measured.numpy())

Treinando ensemble de inferência do valor...


100%|██████████| 1000/1000 [13:28<00:00,  1.24it/s]

Treinamento concluído em 808.59s


In [7]:
modelo_nn.save("modelo_nn_ensamble_tds.pt")

Modelo salvo em: modelo_nn_ensamble_tds.pt


### Modelo treinado com MC do train dataset

In [8]:
modelo_nn_tmc = NNEnsambleModel(model_fn,
                config,
                n_models=n_models_ensemble,
                verbose=True)

modelo_nn_tmc.fit(dataset["train"].X_measured_mc.reshape((-1, 1)).numpy(), 
                  dataset["train"].y_measured_mc.numpy(), 
                  ds_size=dataset["train"].num_amostras)

Treinando ensemble de inferência do valor...


100%|██████████| 1000/1000 [14:12<00:00,  1.17it/s]

Treinamento concluído em 852.34s


In [9]:
modelo_nn_tmc.save("modelo_nn_ensamble_tmc.pt")

Modelo salvo em: modelo_nn_ensamble_tmc.pt


### Modelo Pacheco 

In [10]:
modelo = UQModel(model_fn,
                config,
                n_models=n_models_ensemble,
                mcs_samples=MC_simulation_samples,
                input_std=dataset["train"].erro_aleatorio_x,
                u_M=dataset["train"].erro_aleatorio_y,
                k=2,
                verbose=True)

modelo.fit(dataset["train"].X_measured.reshape((-1, 1)).numpy(),
           dataset["train"].y_measured.numpy())

Treinando ensemble de inferência do valor...


100%|██████████| 1000/1000 [14:56<00:00,  1.12it/s]


Treinando ensemble de incerteza...


100%|██████████| 1000/1000 [14:23<00:00,  1.16it/s]

Treinamento concluído em 1760.12s


In [11]:
modelo.save("modelo_pacheco_tds.pt")

Modelo salvo em: modelo_pacheco_tds.pt


## Modelos de Estimativa

In [5]:
model_dict = {"rf": RFEnsambleModel(dataset["train"].X_measured,
                                    dataset["train"].y_measured,
                                    n_models_ensemble=n_models_ensemble,
                                    num_amostras_treino=dataset["train"].num_amostras),
              "nn": NNEnsambleModel.load("modelo_nn_ensamble_tds.pt"),
              "rf_tmc": RFEnsambleModel(dataset["train"].X_measured_mc,
                                        dataset["train"].y_measured_mc,
                                        n_models_ensemble=n_models_ensemble,
                                        num_amostras_treino=dataset["train"].num_amostras),
              "nn_tmc": NNEnsambleModel.load("modelo_nn_ensamble_tmc.pt"),
              "pacheco": UQModel.load("modelo_pacheco_tds.pt")}

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 799 tasks      | elapsed:    0.5s


Usando dispositivo: cuda


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done 799 tasks      | elapsed:    0.8s


Usando dispositivo: cuda
Usando dispositivo: cuda


In [ ]:
cp_dict = {}
uncertainty_dict = {}
for name, model in model_dict.items():

    print(f"Avaliando modelo {name}:")
    cp_dict[name] = CPCalibration(model,
                                dataset_calib=dataset["train"],
                                mc=not("tmc" in name),
                                alpha=alpha)
    uncertainty_dict[name] = UncertaintyEvaluator(model, 
                                                cp_dict[name].cp_dict,
                                                dataset["test"], 
                                                alpha=alpha, 
                                                verbose=True)
    print("="*50)
    

Avaliando modelo rf:
Model       |   Coverage   |   Average Width
--------------------------------------------------
model       |      59.0000%|      126.3577
absolute    |      97.6000%|      603.4738
normalized  |      98.4000%|      562.6788
monte_carlo |      90.0000%|      255.7800
Avaliando modelo nn:
Model       |   Coverage   |   Average Width
--------------------------------------------------
model       |      29.0000%|       63.2552
absolute    |      97.7000%|      500.9440
normalized  |      98.4000%|      563.5480
monte_carlo |      59.2000%|      177.5027
Avaliando modelo rf_tmc:
Model       |   Coverage   |   Average Width
--------------------------------------------------
model       |      99.3000%|      326.6374
absolute    |      95.2000%|      411.5663
normalized  |      94.1000%|      242.2446
monte_carlo |      77.1000%|      170.0291
Avaliando modelo nn_tmc:
Model       |   Coverage   |   Average Width
--------------------------------------------------
model   

In [ ]:
for name_model in uncertainty_dict.keys():
    for name in uncertainty_dict[name_model].y_pred.keys():
        print(f'Gráfico do modelo {name_model} com {name}:')
        uncertainty_dict[name_model].graph_with_uncertainty(name)